# AMPS I · Week 02 — Lab
## Nodal bases, Basix elements, and Lagrange variants

| | |
|---|---|
| **Lecture sections** | Part A (Vandermonde) · Part B (Basix) · Part C (variants) · Part D (LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | compute nodal bases by hand · inspect and tabulate Basix elements · see why Basix avoids monomials · compare Lagrange variants in DOLFINx |

**Reminder.** `%%fenicsx` cells run in a separate MPI process and are the only cells where `basix`, `ufl`, `dolfinx`
are available. Ordinary cells have NumPy and matplotlib only. Data passes through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The Vandermonde recipe by hand  *(lecture §A3–A4)*
$V_{ij}=N_j(\psi_i)$ with $\psi=\{1,x,x^2\}$; then $\mu=V^{-1}$ and $\phi_i=\sum_j\mu_{ij}\psi_j$.
A nodal variable is just a function that takes the **monomial degree** $j$ and returns $N(x^j)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def nodal_basis_1d(functionals, k):
    V = np.array([[N(j) for N in functionals] for j in range(k + 1)])   # V[i, j] = N_j(x^i)
    return np.linalg.inv(V)

lagrange = [lambda j: 0.0**j, lambda j: 0.5**j, lambda j: 1.0**j]      # p(0), p(1/2), p(1)
integral = [lambda j: 0.0**j, lambda j: 1.0**j, lambda j: 1 / (j + 1)]  # p(0), p(1), ∫p

xs = np.linspace(0, 1, 201)
Psi = np.array([xs**0, xs, xs**2])
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, (name, N) in zip(axs, (("Lagrange", lagrange), ("integral DOF", integral))):
    mu = nodal_basis_1d(N, 2)
    print(f"{name}: mu =\n{np.round(mu, 6) + 0.0}")   # + 0.0 turns -0. into 0.
    # numerical duality check N_i(phi_j) = delta_ij
    D = np.array([[sum(mu[j, l] * Ni(l) for l in range(3)) for j in range(3)] for Ni in N])
    print("   max |N_i(phi_j) - delta_ij| =", np.abs(D - np.eye(3)).max())
    for i in range(3):
        ax.plot(xs, mu[i] @ Psi, label=f"$\\phi_{i}$")
    ax.set_title(name); ax.legend(); ax.grid(alpha=0.3)
plt.show()

> **✏️ Your turn (5 min).** Nodal variables $N=\{p(0),\;p'(0),\;p(1)\}$ on $\mathscr P_2$.
> Write the three functionals (hint: $\frac{d}{dx}x^j\big|_{x=0}$ is 1 for $j=1$ and 0 otherwise), compute and plot the basis.
> Is the element unisolvent? What about $\{p(0),\;p'(\tfrac12),\;p(1)\}$?

In [ ]:
# TODO: functionals for {p(0), p'(0), p(1)} and {p(0), p'(1/2), p(1)}; try nodal_basis_1d

## 2 · Basix elements  *(lecture §B1–B2)*
### 2.1 Inspecting P1–P4 on the triangle

In [ ]:
%%fenicsx -np 1
import basix
import numpy as np
P, T, LV = basix.ElementFamily.P, basix.CellType.triangle, basix.LagrangeVariant

for k in (1, 2, 3, 4):
    e = basix.create_element(P, T, k, LV.equispaced)
    nv, ne, nc = (sum(len(d) for d in e.entity_dofs[dim]) for dim in range(3))
    print(f"P{k}: dim {e.dim:2d} = {nv} vertex + {ne} edge + {nc} interior")

e = basix.create_element(P, T, 2, LV.equispaced)
print("\nP2 points (Basix order):\n", e.points)
print("P2 entity_dofs:", e.entity_dofs)
print("P2 on the interval, points:", basix.create_element(P, basix.CellType.interval, 2, LV.equispaced).points.ravel())

### 2.2 Tabulation and the two sanity checks

In [ ]:
%%fenicsx -np 1
import basix
import numpy as np
P, T, LV = basix.ElementFamily.P, basix.CellType.triangle, basix.LagrangeVariant

e = basix.create_element(P, T, 3, LV.equispaced)
pts, wts = basix.make_quadrature(T, 6)
tab = e.tabulate(1, pts)
print("tabulate(1, pts).shape =", tab.shape, " → (derivative, point, dof, value)")
phi, dx, dy = tab[0, :, :, 0], tab[1, :, :, 0], tab[2, :, :, 0]
print("max |sum phi - 1|  =", np.abs(phi.sum(1) - 1).max())
print("max |sum grad phi| =", max(np.abs(dx.sum(1)).max(), np.abs(dy.sum(1)).max()))
# duality: phi_j at the nodes = identity
print("max |phi_j(x_i) - delta_ij| =", np.abs(e.tabulate(0, e.points)[0, :, :, 0] - np.eye(e.dim)).max())

# save P2 basis on a fine lattice for plotting
e2 = basix.create_element(P, T, 2, LV.equispaced)
lat = basix.create_lattice(T, 40, basix.LatticeType.equispaced, True)
np.savez("/tmp/fnx/w02_p2.npz", lat=lat, phi=e2.tabulate(0, lat)[0, :, :, 0], nodes=e2.points)
print("saved /tmp/fnx/w02_p2.npz")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

d = np.load("/tmp/fnx/w02_p2.npz")
tri = mtri.Triangulation(d["lat"][:, 0], d["lat"][:, 1])
fig, axs = plt.subplots(1, 6, figsize=(16, 2.9))
for j, ax in enumerate(axs):
    c = ax.tricontourf(tri, d["phi"][:, j], levels=np.linspace(-0.15, 1, 24), cmap="viridis")
    ax.plot(*d["nodes"].T, "wo", ms=4); ax.plot(*d["nodes"][j], "r*", ms=12)
    ax.set_aspect("equal"); ax.set_title(f"P2 $\\phi_{j}$"); ax.axis("off")
plt.show()

> **✏️ Your turn (3 min).** Run `basix.create_element(P, T, 3)` (no variant) inside a `try/except` and print the message.
> Then print `basix.ufl.element("Lagrange", "triangle", 3).lagrange_variant`. Which variant does DOLFINx use for `("Lagrange", 3)`?

In [ ]:
%%fenicsx -np 1
# TODO: try P3 without a variant; print the variant DOLFINx uses by default

## 3 · What Basix stores — and why not monomials  *(lecture §B3)*

In [ ]:
%%fenicsx -np 1
import basix
import numpy as np
P, LV, I = basix.ElementFamily.P, basix.LagrangeVariant, basix.CellType.interval

e = basix.create_element(P, basix.CellType.triangle, 2, LV.equispaced)
pts, _ = basix.make_quadrature(basix.CellType.triangle, 4)
Pj = basix.tabulate_polynomials(basix.PolynomialType.legendre, basix.CellType.triangle, 2, pts)
x, y = pts.T
M = np.array([x**0, x, y, x * x, x * y, y * y])
phi = e.tabulate(0, pts)[0, :, :, 0]
print("phi = (C @ orthonormal)^T ? max diff", np.abs(phi - (e.coefficient_matrix @ Pj).T).max())
print("phi = (C @ monomials)^T ?  max diff", np.abs(phi - (e.coefficient_matrix @ M).T).max())

ks = list(range(1, 15))
cm = [np.linalg.cond(np.vander(np.linspace(0, 1, k + 1), increasing=True)) for k in ks]
cb = [np.linalg.cond(basix.create_element(P, I, k, LV.gll_warped).coefficient_matrix) for k in ks]
np.savez("/tmp/fnx/w02_cond.npz", k=ks, mono=cm, basix=cb)
print("saved /tmp/fnx/w02_cond.npz")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
d = np.load("/tmp/fnx/w02_cond.npz")
plt.figure(figsize=(6, 3.8))
plt.semilogy(d["k"], d["mono"], "o-", label="monomial Vandermonde (equispaced)")
plt.semilogy(d["k"], d["basix"], "s-", label="Basix coefficient matrix (GLL)")
plt.xlabel("degree k (interval)"); plt.ylabel("condition number"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 4 · Lagrange variants in DOLFINx  *(lecture Part C)*
Interpolate the Runge function $g(x)=1/(1+25x^2)$ on **one** cell $[-1,1]$ with degree $k$,
using equispaced and GLL-warped nodes.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import basix, basix.ufl, ufl
from dolfinx import fem, mesh

def runge_error(n_cells, k, variant):
    msh = mesh.create_interval(MPI.COMM_SELF, n_cells, [-1.0, 1.0])
    V = fem.functionspace(msh, basix.ufl.element(basix.ElementFamily.P, msh.basix_cell(), k, variant))
    u = fem.Function(V)
    u.interpolate(lambda x: 1 / (1 + 25 * x[0]**2))
    x = ufl.SpatialCoordinate(msh)
    dx = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": 40})
    err = np.sqrt(fem.assemble_scalar(fem.form((u - 1 / (1 + 25 * x[0]**2))**2 * dx)))
    return err, V.dofmap.index_map.size_global

ks = [2, 4, 6, 8, 10, 12, 14, 16]
out = {}
for v in (basix.LagrangeVariant.equispaced, basix.LagrangeVariant.gll_warped):
    out[v.name] = [runge_error(1, k, v)[0] for k in ks]
np.savez("/tmp/fnx/w02_runge.npz", k=ks, **out)
for k, a, b in zip(ks, out["equispaced"], out["gll_warped"]):
    print(f"k={k:2d}: equispaced {a:.3e}   gll_warped {b:.3e}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
d = np.load("/tmp/fnx/w02_runge.npz")
plt.figure(figsize=(6, 3.8))
plt.semilogy(d["k"], d["equispaced"], "o-", label="equispaced")
plt.semilogy(d["k"], d["gll_warped"], "s--", label="gll_warped (DOLFINx default)")
plt.xlabel("degree k (one cell)"); plt.ylabel("$L^2$ interpolation error"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

> **✏️ Your turn (10 min).** Same number of unknowns, different strategy: compare **one cell of degree 16** (17 DOFs)
> with **4 cells of degree 4** and **16 cells of degree 1** (17 DOFs each), for both variants.
> Which wins for the Runge function, and why? *(Copy `runge_error` into the cell — variables do not survive between `%%fenicsx` cells.)*

In [ ]:
%%fenicsx -np 1
# TODO: runge_error for (1 cell, k=16), (4 cells, k=4), (16 cells, k=1), both variants; print errors and DOF counts

## 5 · LLM-assisted coding: live demo  *(lecture Part D)*
Prompt (paste the API reference sheet first):

```text
Context: [API reference sheet]
Task: Basix 0.11 code for a %%fenicsx -np 1 cell. Create the degree-3 Lagrange element on the reference
triangle, tabulate all basis functions on a fine lattice, and save them with the node coordinates to
/tmp/fnx/p3.npz. In a separate ordinary cell (NumPy + matplotlib only), plot the 10 basis functions in a
2×5 grid; label each subplot "vertex", "edge" or "interior" using the element's entity_dofs.
```
Paste the result below **unchanged**, run it, then go through the checklist.

In [ ]:
%%fenicsx -np 1
# TODO: paste the %%fenicsx part your LLM produced (unchanged) and run it

**Verification checklist**

- [ ] A variant is given explicitly (degree 3 needs one)
- [ ] The tabulate output is indexed `[0, :, :, 0]` → shape `(n_points, 10)`, not transposed
- [ ] The labels come from `entity_dofs`, not from an assumed ordering
- [ ] Each plot has its maximum 1 at its own node and 0 at the other nine nodes
- [ ] No `basix` import in the ordinary plotting cell (it would fail on Colab)